# SWEFil epoch100 — frozen-model validation experiments on T4

Run the single cell below in a Colab **T4 GPU** runtime. It evaluates the completed epoch100 model on the existing **91 validation images / 101 annotation groups**, with no additional training.

The fixed comparisons are the reviewed native recipe, four-view flip TTA, more uniform overlapping tile placement, and raw input without limb correction. Each uses the same grid of foreground thresholds and minimum component areas. Results are development comparisons on a previously used validation set, not an independent holdout estimate.

Completed probability maps are saved on Drive and reused after interruption. Run the **same cell** to resume; an unfinished image is repeated. Dependencies and the verified 128.5 MB checkpoint may be downloaded again after a runtime reset. Cache storage can require several GB. Existing training and submission folders are kept separate.

The first recipe checks that the original validation PQ can be reproduced. A discrepancy greater than 0.003 stops the experiment for diagnosis. The finished review ZIP contains metrics and validation predictions; upload that ZIP for analysis. This notebook does not generate or submit a test CSV.

Progress appears periodically and the complete log is saved under `fixed_split_experiments/swefil_epoch100_inference_ablation_t4_v1/inference.log`. Use the same Google account for Drive mounting and authentication.


In [ ]:
# Frozen epoch100 validation ablations. Run this same cell again to resume.
from pathlib import Path
from google.colab import auth, drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
TRAINING_ROOT = BASE / "fixed_split_experiments/swefil_all_mixed_native_t4_v1"
OUTPUT = BASE / "fixed_split_experiments/swefil_epoch100_inference_ablation_t4_v1"
RECIPES = "baseline,tta4,dense,raw"
CHECKPOINT_FILE_ID = None  # Normally read automatically from completed training metadata.
CHECKPOINT = None  # Optional exact reviewed epoch100 local weight file.
CHECKOUT = Path("/content/solar_swefil_ablation_github")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
drive.mount("/content/drive")
if CHECKPOINT is None:
    auth.authenticate_user()
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
pointer = OUTPUT / "ablation_bootstrap_state.json"
if pointer.exists() or pointer.is_symlink():
    if pointer.is_symlink():
        raise ValueError("Unsafe saved bootstrap pointer.")
    state = json.loads(pointer.read_text())
    if state.get("scope") != "swefil_epoch100_inference_ablation_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("Saved source belongs to another experiment.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed.")
    manifest = json.loads(manifest_path.read_text())
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(canonical).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    for name, expected in manifest.items():
        path = snapshot / name
        if path.is_symlink() or not path.resolve().is_relative_to(snapshot.resolve()) or not path.is_file() or sha(path) != expected:
            raise ValueError("Saved source changed: " + name)
    bootstrap = snapshot / "tools/swefil_ablation_colab.py"
    if "tools/swefil_ablation_colab.py" not in manifest:
        raise ValueError("The saved manifest has no ablation launcher.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Preserve this checkout and choose another CHECKOUT path.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/swefil_ablation_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("This code version lacks the validation ablation launcher.")
command = [sys.executable, "-u", str(bootstrap), "--training-root", str(TRAINING_ROOT),
           "--output", str(OUTPUT), "--recipes", RECIPES]
if CHECKPOINT is not None: command += ["--checkpoint", str(CHECKPOINT)]
if CHECKPOINT_FILE_ID is not None: command += ["--checkpoint-file-id", str(CHECKPOINT_FILE_ID)]
process = subprocess.Popen(command, start_new_session=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                           errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result = json.loads((OUTPUT / "latest_result.json").read_text())
expected = dict(status="complete", validation_images=91, annotation_groups=101, gt_instances=708,
                checkpoint_epoch=100, checkpoint_sha256="2b22ed327e2f64902bb7b9b3984586d1ba893f52ada74f557622cce4dac27e40")
if any(result.get(key) != value for key, value in expected.items()) or Path(result.get("output_root", "")).resolve() != OUTPUT:
    raise RuntimeError("A complete reviewed epoch100 validation run is required before download.")
archive = Path(result["archive"])
if archive.is_symlink() or not archive.resolve().is_relative_to(OUTPUT) or not archive.is_file() or sha(archive) != result.get("archive_sha256"):
    raise ValueError("Review archive is missing or changed.")
print("Upload this review ZIP after download:", archive)
files.download(str(archive))
